# Confidence-probe diagnostic: prefill-probe stopping cascade
Results reported in the paper (Table 14) are in `results/confidence_probe/`. Paste these cells **below the existing cells** of `rl_ar_pipeline.ipynb`, in the same Kaggle session that holds
`/kaggle/working/rlar` with the caches of all (model, dataset) pairs. Then run them top to bottom.
Nothing has to be edited per model or dataset: every pair with a cache is found and processed in a loop.

* **Stage 0 (gate):** prefill probes for `GATE_CONFIGS` only (no latency), then the G0 gate.
* **Stage 1 (automatic if the gate says GO and `AUTO_CONTINUE=True`):** probes, reranker scores and latency
  for every pair, then all experiment blocks.

Every step is checkpointed and skipped on re-run, so if the session dies just run all cells again.
These cells redefine some helper names of the pipeline notebook (`evaluate`, `MLP`, `holm`, ...):
do not re-run the earlier unified cells afterwards without restarting the kernel.

In [ ]:
# ============================== CONFIG (the only cell to edit, if at all) ==============================
OUT_ROOT     = "/kaggle/working/rlar"
EXTRA_SRC    = []                                   # other roots with data/ and runs/ (e.g. attached Kaggle inputs)
PS_DATASETS  = ["nq", "squad", "hotpotqa"]
PS_MODELS    = ["llama1b", "llama3b", "qwen3b", "mistral7b", "llama8b"]
GATE_CONFIGS = [("nq", "llama1b"), ("squad", "llama1b"), ("hotpotqa", "llama1b")]   # cheap pairs for stage 0
AUTO_CONTINUE = True                                # GO at the gate -> stage 1 runs by itself

# probe pass (GPU)
MAX_NEW_TOKENS = 32
PROBE_BATCH    = {0: 32, 1: 16, 2: 12, 3: 8}        # auto-halves on OOM
PROBE_CHUNK    = 256
TOPK_STORE     = 10
STORE_HIDDEN   = True
LAT_N_TEST, LAT_N_VAL = 150, 60
RERANK_REPO    = "BAAI/bge-reranker-base"

# experiments (CPU)
SEEDS       = [0, 1, 2, 3, 4]
FRONT_SEEDS = [0, 1, 2]
K_OPTIONS   = [0, 1, 2, 3]
W_EM        = 0.5
LAMBDA_MAIN = 0.10
LAMBDA_GRID = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.75]
LAT_LAMBDA_MAIN = 0.10
BUDGET_FRACS = [0.50, 0.60, 0.70, 0.80, 0.90]
HID_DIM     = 32
POLICY_HIDDEN = [128, 128]
CPPO = dict(episodes=100_000, rollout_eps=1024, epochs=10, minibatch=1024, lr=3e-4, clip=0.2, ent=0.01, vf=0.5, gae=0.95)
CRF  = dict(episodes=100_000, batch_eps=256, lr=3e-4, ent=0.01, vf=0.5)
CFI  = dict(max_epochs=400, lr=1e-3, batch=256, wd=1e-4, patience=30, ent=0.01)
BPPO = dict(total_samples=200_000, rollout=4096, epochs=10, minibatch=512, lr=3e-4, clip=0.2, ent=0.01, vf=0.5)
FRONT_EPISODES = 50_000
INTERACTION_BUDGETS = [500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000]
N_BOOT = 10_000
RUN = {k: (k == "G0") for k in ["G0", "P1", "P2", "P3", "P4", "P5", "P6", "S"]}   # stage 1 switches the rest on
IGNORE_GATE = False
FORCE = False
QUICK = False
# ==========================================================================================================

In [ ]:
# ---------------- prefill-probe cache: one function, looped over (model, dataset) ----------------
import os, json, time, re, string, copy, gc, glob
import numpy as np, torch
from tqdm.auto import tqdm

PS_REGISTRY = {
    "llama1b":   dict(repo="meta-llama/Llama-3.2-1B-Instruct",   quant=None),
    "llama3b":   dict(repo="meta-llama/Llama-3.2-3B-Instruct",   quant=None),
    "qwen3b":    dict(repo="Qwen/Qwen2.5-3B-Instruct",           quant=None),
    "llama8b":   dict(repo="meta-llama/Llama-3.1-8B-Instruct",   quant="nf4"),
    "mistral7b": dict(repo="mistralai/Mistral-7B-Instruct-v0.3", quant="nf4"),
}
PC_DEV = "cuda" if torch.cuda.is_available() else "cpu"
PC_ROOTS = [OUT_ROOT] + list(EXTRA_SRC) + sorted(glob.glob("/kaggle/input/*/rlar")) + sorted(glob.glob("/kaggle/input/*/*/rlar"))

def pc_find(rel, required=True):
    for r in PC_ROOTS:
        p = os.path.join(r, rel)
        if os.path.exists(p): return p
    if required: raise FileNotFoundError(f"{rel} not found in {PC_ROOTS}")
    return None
def pc_json(p, default=None):
    if p is None or not os.path.exists(p): return default
    with open(p) as f: return json.load(f)
def pc_save_json(o, p):
    t = p + ".tmp"
    with open(t, "w") as f: json.dump(o, f, indent=1, default=lambda x: x.item() if hasattr(x, "item") else str(x))
    os.replace(t, p)
def pc_jsonl(p):
    if p is None or not os.path.exists(p): return []
    out = []
    with open(p) as f:
        for l in f:
            l = l.strip()
            if l:
                try: out.append(json.loads(l))
                except json.JSONDecodeError: pass
    return out
def pc_append(p, rows):
    with open(p, "a") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
        f.flush(); os.fsync(f.fileno())
def pc_savez(p, **a): t = p + ".tmp.npz"; np.savez(t, **a); os.replace(t, p)
_PUN = set(string.punctuation)
def pc_norm(s):
    s = "".join(ch for ch in s.lower() if ch not in _PUN); s = re.sub(r"\b(a|an|the)\b", " ", s); return " ".join(s.split())

PC_SYSTEM = ("You are a question answering assistant. Answer with the shortest possible span "
             "(at most 5 words). Output only the answer, with no explanation.")
def pc_messages(q, passages):
    if passages:
        docs = "\n\n".join(f"[{i+1}] {p['title']}: {p['text']}" for i, p in enumerate(passages))
        user = f"Use the documents if they are helpful.\n\nDocuments:\n{docs}\n\nQuestion: {q}"
    else:
        user = f"Question: {q}"
    return [{"role": "system", "content": PC_SYSTEM}, {"role": "user", "content": user}]
def pc_render(tok, q, passages=None):
    m = pc_messages(q, passages)
    try: return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, date_string="26 Jul 2024")
    except Exception:
        return tok.apply_chat_template([{"role": "user", "content": m[0]["content"] + "\n\n" + m[1]["content"]}],
                                       tokenize=False, add_generation_prompt=True)
def pc_clean(t):
    t = t.strip().split("\n")[0].strip(); t = re.sub(r"^(final answer|answer)\s*[:\-]\s*", "", t, flags=re.I)
    return re.sub(r"[.。]+$", "", t.strip(" \"'`*")).strip()
def pc_lcp(a, b):
    n = min(len(a), len(b)); i = 0
    while i < n and a[i] == b[i]: i += 1
    return i

_PC_DATA = {}
def pc_data(ds):
    if ds not in _PC_DATA:
        rel = f"data/{ds}"; sp = pc_json(pc_find(f"{rel}/splits.json")); P = pc_jsonl(pc_find(f"{rel}/corpus.jsonl"))
        ret = {s: np.load(pc_find(f"{rel}/retrieval_{s}.npz"))["idx"] for s in sp}
        _PC_DATA[ds] = dict(splits=sp, P=P, pid2idx={p["pid"]: i for i, p in enumerate(P)}, idx=ret)
    return _PC_DATA[ds]

def pc_free_unified_llm():
    g = globals()
    if g.get("_LLM") is not None:
        try: del g["_LLM"].model
        except Exception: pass
        g["_LLM"] = None
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

def pc_rerank(ds):
    """Cross-encoder scores of the top-3 passages; model-independent, once per dataset."""
    out = f"{OUT_ROOT}/data/{ds}"; os.makedirs(out, exist_ok=True); D = pc_data(ds)
    ms_path = f"{out}/rerank_ms.json"
    if all(os.path.exists(f"{out}/rerank_{s}.npy") for s in D["splits"]) and os.path.exists(ms_path): return
    from sentence_transformers import CrossEncoder
    from huggingface_hub import HfApi
    pins = pc_json(f"{OUT_ROOT}/pins.json", {}); sha = pins.get(f"model:{RERANK_REPO}") or HfApi().model_info(RERANK_REPO).sha
    pins[f"model:{RERANK_REPO}"] = sha; pc_save_json(pins, f"{OUT_ROOT}/pins.json")
    CE = CrossEncoder(RERANK_REPO, revision=sha, device=PC_DEV, max_length=512)
    pair = lambda q, i: (q, f"{D['P'][i]['title']}: {D['P'][i]['text']}")
    for s, items in D["splits"].items():
        if os.path.exists(f"{out}/rerank_{s}.npy"): continue
        pr = [pair(it["question"], int(i)) for j, it in enumerate(items) for i in D["idx"][s][j, :3]]
        np.save(f"{out}/rerank_{s}.npy", np.asarray(CE.predict(pr, batch_size=64, show_progress_bar=False), np.float32).reshape(-1, 3))
    ts = []
    for j, it in enumerate(D["splits"]["test"][:LAT_N_TEST]):
        pr = [pair(it["question"], int(i)) for i in D["idx"]["test"][j, :3]]
        if PC_DEV == "cuda": torch.cuda.synchronize()
        t = time.perf_counter(); CE.predict(pr, batch_size=3, show_progress_bar=False)
        if PC_DEV == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t) * 1000)
    pc_save_json(dict(repo=RERANK_REPO, revision=sha, ms_mean=float(np.mean(ts[3:]))), ms_path)
    del CE; gc.collect(); torch.cuda.empty_cache()

def probe_config(ds, mk, do_latency=False):
    """Prefill probes for depths 0..3 on all splits (+ optional one-engine latency profile) for one pair."""
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    run_rel = f"runs/{ds}/{mk}"; pdir = f"{OUT_ROOT}/{run_rel}/probe"; os.makedirs(f"{pdir}/parts", exist_ok=True)
    D = pc_data(ds); SPL_ = D["splits"]; Pas = D["P"]; IDX = D["idx"]
    lat_need = do_latency and any(len(pc_jsonl(f"{pdir}/latency_{s}.jsonl")) < n for s, n in [("test", LAT_N_TEST), ("val", LAT_N_VAL)])
    if all(os.path.exists(f"{pdir}/probe_{s}.npz") for s in SPL_) and not lat_need:
        print(f"[skip] {ds}/{mk}: probe cache complete"); return
    repo = PS_REGISTRY[mk]["repo"]; pins = pc_json(pc_find("pins.json", False), {})
    sha = pins.get(f"model:{repo}") or pc_json(pc_find(f"{run_rel}/manifest.json", False), {}).get("model_revision")
    assert sha, f"no pinned revision for {repo}; refusing to load an unpinned model"
    pc_free_unified_llm()
    tok = AutoTokenizer.from_pretrained(repo, revision=sha, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(revision=sha, device_map={"": 0}, torch_dtype=torch.float16)
    if PS_REGISTRY[mk]["quant"] == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    lm = AutoModelForCausalLM.from_pretrained(repo, **kw).eval()
    eos = lm.generation_config.eos_token_id; eos = eos if isinstance(eos, list) else [eos]
    stop_ids = {int(e) for e in eos if e is not None} | {int(tok.eos_token_id), int(tok.pad_token_id)}
    layers = next(m for m in lm.modules() if isinstance(m, torch.nn.ModuleList) and len(m) == lm.config.num_hidden_layers)
    hl = {"mid": len(layers) // 2, "last": len(layers) - 1}; hook = {}; handles = []
    def mk_hook(name):
        def h(mod, inp, out):
            if hook.get("_on"):
                x = out[0] if isinstance(out, (tuple, list)) else out; hook[name] = x[:, -1, :].detach().to(torch.float16).cpu()
        return h
    if STORE_HIDDEN: handles = [layers[i].register_forward_hook(mk_hook(n)) for n, i in hl.items()]
    def fwd(ids, am=None, cache=None, use_cache=False, pos=None):
        a = dict(input_ids=ids, use_cache=use_cache)
        if am is not None: a["attention_mask"] = am
        if pos is not None: a["position_ids"] = pos
        if cache is not None: a["past_key_values"] = cache
        for extra in ({"logits_to_keep": 1}, {"num_logits_to_keep": 1}, {}):
            try: return lm(**a, **extra)
            except TypeError: continue
        raise RuntimeError("forward failed")
    ctx = lambda s, j, k: [Pas[i] for i in IDX[s][j, :k]] if k > 0 else None
    try:
        # ---- probe features ----
        @torch.inference_mode()
        def batch(prompts):
            enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(lm.device)
            am = enc["attention_mask"]; pos = (am.long().cumsum(-1) - 1).clamp(min=0)
            hook.clear(); hook["_on"] = STORE_HIDDEN; out = fwd(enc["input_ids"], am, pos=pos); hook["_on"] = False
            lp = torch.log_softmax(out.logits[:, -1].float(), -1); p = lp.exp(); top = p.topk(TOPK_STORE, -1)
            r = dict(maxp=top.values[:, 0].cpu().numpy(), margin=(top.values[:, 0] - top.values[:, 1]).cpu().numpy(),
                     ent=(-(p * lp.clamp(min=-1e4)).sum(-1)).cpu().numpy(), top_ids=top.indices.cpu().numpy().astype(np.int32),
                     top_p=top.values.cpu().numpy().astype(np.float16), n_in=am.sum(1).cpu().numpy().astype(np.int32))
            if STORE_HIDDEN:
                for n in hl: r[f"hid_{n}"] = hook[n].numpy()
            return r
        def many(prompts, bs):
            order = np.argsort([len(p) for p in prompts])[::-1]; out = [None] * len(prompts); i, cur = 0, bs
            while i < len(order):
                ix = order[i:i + cur]
                try: r = batch([prompts[j] for j in ix])
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if cur == 1: raise
                    cur = max(1, cur // 2); continue
                for b, j in enumerate(ix): out[j] = {k_: v[b] for k_, v in r.items()}
                i += len(ix)
            return {k_: np.stack([o[k_] for o in out]) for k_ in out[0]}
        for s in ["test", "val", "train"]:
            final = f"{pdir}/probe_{s}.npz"
            if os.path.exists(final): continue
            items = SPL_[s]; n = len(items)
            for k in K_OPTIONS:
                for c0 in tqdm(range(0, n, PROBE_CHUNK), desc=f"{ds}/{mk} probe {s} k={k}", leave=False):
                    part = f"{pdir}/parts/{s}_k{k}_{c0:05d}.npz"
                    if os.path.exists(part): continue
                    js = range(c0, min(n, c0 + PROBE_CHUNK))
                    pc_savez(part, **many([pc_render(tok, items[j]["question"], ctx(s, j, k)) for j in js], PROBE_BATCH[k]))
            lcp, lcp0 = np.zeros((n, len(K_OPTIONS)), np.int32), np.zeros((n, len(K_OPTIONS)), np.int32)
            for j, it in enumerate(items):
                ids = [tok(pc_render(tok, it["question"], ctx(s, j, k)), add_special_tokens=False)["input_ids"] for k in K_OPTIONS]
                for ki in range(1, len(K_OPTIONS)): lcp[j, ki] = pc_lcp(ids[ki - 1], ids[ki]); lcp0[j, ki] = pc_lcp(ids[0], ids[ki])
            merged = {}
            for k in K_OPTIONS:
                parts = [np.load(f"{pdir}/parts/{s}_k{k}_{c0:05d}.npz") for c0 in range(0, n, PROBE_CHUNK)]
                for key in parts[0].files: merged.setdefault(key, []).append(np.concatenate([p_[key] for p_ in parts], 0))
            pc_savez(final, qids=np.array([it["qid"] for it in items]), lcp=lcp, lcp0=lcp0, **{k_: np.stack(v, 1) for k_, v in merged.items()})
        # ---- one-engine latency profile ----
        if do_latency:
            def sync():
                if PC_DEV == "cuda": torch.cuda.synchronize()
            def timed(fn):
                sync(); t = time.perf_counter(); r = fn(); sync(); return r, (time.perf_counter() - t) * 1000.0
            ids_of = lambda q, ps: tok(pc_render(tok, q, ps), return_tensors="pt", add_special_tokens=False)["input_ids"].to(lm.device)
            @torch.inference_mode()
            def pre_full(q, ps):
                ids = ids_of(q, ps); o = fwd(ids, use_cache=True); return ids, o.past_key_values, o.logits[:, -1]
            @torch.inference_mode()
            def pre_inc(q, ps, cache, keep):
                ids = ids_of(q, ps); keep = min(keep, ids.shape[1] - 1); cache.crop(keep)
                o = fwd(ids[:, keep:], cache=cache, use_cache=True); return ids, o.past_key_values, o.logits[:, -1]
            @torch.inference_mode()
            def decode(cache, logits):
                t_ = logits.argmax(-1); out = []
                for _ in range(MAX_NEW_TOKENS):
                    if int(t_[0]) in stop_ids: break
                    out.append(int(t_[0])); o = fwd(t_.view(1, 1), cache=cache, use_cache=True); cache = o.past_key_values; t_ = o.logits[:, -1].argmax(-1)
                return pc_clean(tok.decode(out, skip_special_tokens=True)), len(out) + (1 if len(out) < MAX_NEW_TOKENS else 0)
            maxp = lambda lg: float(torch.softmax(lg.float(), -1).max())
            old = {r["qid"]: r for r in pc_jsonl(pc_find(f"{run_rel}/latency_test.jsonl", False))}
            for s, nq in [("test", LAT_N_TEST), ("val", LAT_N_VAL)]:
                path = f"{pdir}/latency_{s}.jsonl"; items = SPL_[s][:nq]; done = {r["qid"] for r in pc_jsonl(path)}
                todo = [j for j, it in enumerate(items) if it["qid"] not in done]
                if not todo: continue
                G = {k: {r["qid"]: r for r in pc_jsonl(pc_find(f"{run_rel}/gen/{s}_k{k}.jsonl", False))} for k in K_OPTIONS}
                GF = {r["qid"]: r for r in pc_jsonl(pc_find(f"{run_rel}/gen/{s}_flare.jsonl", False))}
                for _ in range(3): pre_full("What is the capital of France?", None)
                buf = []
                for j in tqdm(todo, desc=f"{ds}/{mk} latency {s}", leave=False):
                    it = items[j]; q = it["question"]; row = dict(qid=it["qid"]); o_ = old.get(it["qid"], {})
                    for key in ["t_embed", "t_search", "t_flare_embed", "t_flare_search"]: row[key] = o_.get(key)
                    caches, lgs, ids_all = {}, {}, {}
                    for k in K_OPTIONS:
                        (ids, cache, lg), row[f"t_pre_full_{k}"] = timed(lambda: pre_full(q, ctx(s, j, k)))
                        caches[k], lgs[k], ids_all[k] = cache, lg, ids[0].tolist(); row[f"n_in_{k}"] = len(ids_all[k])
                    for k in K_OPTIONS[1:]:
                        keep = pc_lcp(ids_all[k - 1], ids_all[k]); src = copy.deepcopy(caches[k - 1])
                        (_, _, lg), row[f"t_pre_inc_{k}"] = timed(lambda: pre_inc(q, ctx(s, j, k), src, keep))
                        row[f"lcp_{k}"] = keep; row[f"dmaxp_inc_{k}"] = abs(maxp(lg) - maxp(lgs[k]))
                    for k in K_OPTIONS[2:]:
                        keep = pc_lcp(ids_all[0], ids_all[k]); src = copy.deepcopy(caches[0])
                        _, row[f"t_pre_jump_{k}"] = timed(lambda: pre_inc(q, ctx(s, j, k), src, keep))
                    for k in K_OPTIONS:
                        src = copy.deepcopy(caches[k]); (ans, no), row[f"t_dec_{k}"] = timed(lambda: decode(src, lgs[k]))
                        row[f"n_out_{k}"] = no; cg = G[k].get(it["qid"], {}).get("answer")
                        row[f"match_{k}"] = None if cg is None else float(pc_norm(ans) == pc_norm(cg))
                    fr = GF.get(it["qid"])
                    if fr is not None:
                        fp = [Pas[D["pid2idx"][p_]] for p_ in fr["flare_pids"] if p_ in D["pid2idx"]]
                        (_, cache, lg), row["t_flare_pre"] = timed(lambda: pre_full(q, fp))
                        _, row["t_flare_dec"] = timed(lambda: decode(cache, lg))
                    caches = None; buf.append(row)
                    if len(buf) >= 10: pc_append(path, buf); buf = []
                if buf: pc_append(path, buf)
        z = np.load(f"{pdir}/probe_val.npz")
        print(f"[done] {ds}/{mk}: mean first-token max prob by depth (val) {np.round(z['maxp'].mean(0), 3)}")
    finally:
        for h in handles: h.remove()
        lm = None; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

def available_pairs():
    return [(ds, mk) for ds in PS_DATASETS for mk in PS_MODELS
            if pc_find(f"runs/{ds}/{mk}/gen/train_k3.jsonl", False) and pc_find(f"data/{ds}/retrieval_train.npz", False)]
print("pairs with a generation cache:", available_pairs())

In [ ]:
# ---------------- stage 0: probes for the gate pairs only (no latency, no reranker) ----------------
t0 = time.time()
for ds, mk in GATE_CONFIGS:
    if (ds, mk) in available_pairs(): probe_config(ds, mk, do_latency=False)
    else: print(f"[missing] {ds}/{mk}: no generation cache in this session")
print(f"stage-0 probes: {time.time()-t0:.0f}s")

In [ ]:
import os, json, time, math, copy, pickle, random, warnings, glob
from types import SimpleNamespace
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy import stats as sps
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore", category=RuntimeWarning)
torch.set_num_threads(max(1, os.cpu_count() or 2))
try: display
except NameError: display = print
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)

if QUICK:
    SEEDS, FRONT_SEEDS = [0, 1], [0]
    CPPO.update(episodes=6_000, rollout_eps=512); CRF.update(episodes=6_000)
    CFI.update(max_epochs=20, patience=5); BPPO.update(total_samples=8_000, rollout=1024)
    FRONT_EPISODES = 4_000; INTERACTION_BUDGETS = [500, 2_000]; LAMBDA_GRID = [0.0, 0.1, 0.3]; N_BOOT = 500

STUDY = f"{OUT_ROOT}/probe_study"; os.makedirs(STUDY, exist_ok=True)
SRC_ROOTS = [OUT_ROOT] + list(EXTRA_SRC) + sorted(glob.glob("/kaggle/input/*/rlar")) + \
            sorted(glob.glob("/kaggle/input/*/*/rlar")) + sorted(glob.glob("/kaggle/input/*/*/*/rlar"))
SPL = ["train", "val", "test"]; K = len(K_OPTIONS); STOP, NEXT = 0, 1
WH = ["who", "what", "when", "where", "which", "how", "why"]
UNC = ["first_maxp", "first_ent", "first_margin", "mean_logp", "min_logp", "draft_len"]
RETN = ["s1", "s1_minus_s2", "mean_s1_3", "s1_minus_s10"]
QRY = ["n_words", "cap_ratio", "has_digit"] + [f"wh_{w}" for w in WH] + ["wh_other"]
FNAMES = UNC + RETN + QRY
MNAME = {"llama1b": "Llama-3.2-1B", "llama3b": "Llama-3.2-3B", "qwen3b": "Qwen2.5-3B",
         "mistral7b": "Mistral-7B-v0.3", "llama8b": "Llama-3.1-8B"}
DNAME = {"nq": "NQ", "squad": "SQuAD", "hotpotqa": "HotpotQA"}

def find(rel):
    for r in SRC_ROOTS:
        p = os.path.join(r, rel)
        if os.path.exists(p): return p
    return None

def save_pkl(o, p):
    os.makedirs(os.path.dirname(p), exist_ok=True); t = p + ".tmp"
    with open(t, "wb") as f: pickle.dump(o, f)
    os.replace(t, p)
def load_pkl(p):
    with open(p, "rb") as f: return pickle.load(f)
def read_jsonl(p):
    if p is None or not os.path.exists(p): return []
    out = []
    with open(p) as f:
        for l in f:
            l = l.strip()
            if l:
                try: out.append(json.loads(l))
                except json.JSONDecodeError: pass
    return out
def cached(path, fn):
    if os.path.exists(path) and not FORCE: return load_pkl(path)
    r = fn(); save_pkl(r, path); return r
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
def tag(x): return f"{x:.3f}".replace(".", "p")

## Loading a configuration

In [ ]:
def load_cfg(ds, mk):
    run, dat = f"runs/{ds}/{mk}", f"data/{ds}"
    need = [f"{run}/tables_{s}.pkl" for s in SPL] + [f"{dat}/retrieval_{s}.npz" for s in SPL] + \
           [f"{run}/probe/probe_{s}.npz" for s in SPL]
    if any(find(p) is None for p in need): return None
    C = SimpleNamespace(ds=ds, mk=mk, out=f"{STUDY}/runs/{ds}/{mk}", cache={})
    os.makedirs(C.out, exist_ok=True)
    C.TAB = {s: load_pkl(find(f"{run}/tables_{s}.pkl")) for s in SPL}
    C.RS = {s: np.load(find(f"{dat}/retrieval_{s}.npz"))["scores"].astype(np.float64) for s in SPL}
    C.PR = {}
    for s in SPL:
        z = np.load(find(f"{run}/probe/probe_{s}.npz"))
        assert list(z["qids"]) == list(C.TAB[s]["qids"]), f"question order mismatch in {ds}/{mk}/{s}"
        C.PR[s] = {k: z[k] for k in z.files}
    C.RR = {s: (np.load(find(f"{dat}/rerank_{s}.npy")) if find(f"{dat}/rerank_{s}.npy") else None) for s in SPL}
    rrms = find(f"{dat}/rerank_ms.json"); C.RR_MS = json.load(open(rrms))["ms_mean"] if rrms else np.nan
    tr = C.TAB["train"]
    C.MU, C.SD = tr["X"].mean(0), tr["X"].std(0) + 1e-6
    C.T_REF = float(np.mean(tr["NIN"][:, -1] - tr["NIN"][:, 0]))
    C.nin_match = {s: float((C.PR[s]["n_in"] == C.TAB[s]["NIN"]).mean()) for s in SPL}
    C.LATROWS = {s: read_jsonl(find(f"{run}/probe/latency_{s}.jsonl")) for s in ["test", "val"]}
    C.out = f"{STUDY}/runs/{ds}/{mk}/{'lat' if C.LATROWS['test'] else 'nolat'}"; os.makedirs(C.out, exist_ok=True)
    C.OLDRES = {}
    for nm in ["main_ppo", "main_ppo_nodraft"]:
        p = find(f"{run}/results/{nm}.pkl")
        if p: C.OLDRES[nm] = {sd: np.asarray(ev["k"]).astype(int) for sd, ev in load_pkl(p).items()}
    build_derived(C)
    return C

def configs():
    out = []
    for ds in PS_DATASETS:
        for mk in PS_MODELS:
            run = f"runs/{ds}/{mk}"
            if find(f"{run}/tables_test.pkl") and find(f"{run}/probe/probe_test.npz"):
                out.append((ds, mk))
    return out

## Costs, latency engine, observations

In [ ]:
def build_derived(C):
    for s in SPL:
        T, P = C.TAB[s], C.PR[s]
        NIN, NOUT = T["NIN"].astype(float), T["NOUT"].astype(float)
        inc = NIN.copy(); inc[:, 1:] = NIN[:, 1:] - P["lcp"][:, 1:]
        D = dict(Q=T["F1"] + W_EM * T["EM"], F1=T["F1"], EM=T["EM"], NIN=NIN, NOUT=NOUT, inc=inc)
        # processed prompt tokens when stopping at depth j (prefix reuse), for start depth 0 and 1, and without reuse
        D["proc0"] = np.cumsum(inc, 1)
        p1 = np.full_like(NIN, np.nan); p1[:, 1] = NIN[:, 1]
        for j in range(2, K): p1[:, j] = p1[:, j - 1] + inc[:, j]
        D["proc1"] = p1
        D["proc0_noreuse"] = np.cumsum(NIN, 1)
        D["proc_static"] = NIN
        jump = NIN - P["lcp0"]; jump[:, 0] = 0; D["proc_jump"] = NIN[:, :1] + jump      # k=0 probe, then jump to k
        C.cache[s] = D
    fit_latency_model(C); build_test_latency(C)

def _lat_arrays(rows, key):
    return np.array([r.get(key) if r.get(key) is not None else np.nan for r in rows], float)

def build_test_latency(C):
    """Measured per-question latency components (single timing engine) aligned to the test split."""
    rows = {r["qid"]: r for r in C.LATROWS["test"]}; q = C.TAB["test"]["qids"]
    al = [rows.get(x, {}) for x in q]; L = {}
    keys = [f"t_pre_full_{k}" for k in K_OPTIONS] + [f"t_pre_inc_{k}" for k in K_OPTIONS[1:]] + \
           [f"t_pre_jump_{k}" for k in K_OPTIONS[2:]] + [f"t_dec_{k}" for k in K_OPTIONS] + \
           ["t_embed", "t_search", "t_flare_embed", "t_flare_search", "t_flare_pre", "t_flare_dec"] + \
           [f"match_{k}" for k in K_OPTIONS]
    for kk in keys: L[kk] = _lat_arrays(al, kk)
    prof = np.isfinite(L["t_pre_full_0"])
    for kk in ["t_embed", "t_search", "t_flare_embed", "t_flare_search"]:
        med = np.nanmedian(L[kk]) if np.isfinite(L[kk]).any() else 0.0
        L[kk] = np.where(prof & ~np.isfinite(L[kk]), med, L[kk])
    L["pf"] = np.stack([L[f"t_pre_full_{k}"] for k in K_OPTIONS], 1)
    L["pinc"] = L["pf"].copy()
    for k in K_OPTIONS[1:]: L["pinc"][:, k] = L[f"t_pre_inc_{k}"]
    L["pjump"] = L["pf"].copy(); L["pjump"][:, 1] = L["t_pre_inc_1"]
    for k in K_OPTIONS[2:]: L["pjump"][:, k] = L[f"t_pre_jump_{k}"]
    L["dec"] = np.stack([L[f"t_dec_{k}"] for k in K_OPTIONS], 1)
    L["ret"] = L["t_embed"] + L["t_search"]; L["prof"] = prof
    C.L = L
    C.match_rate = {k: float(np.nanmean(L[f"match_{k}"])) for k in K_OPTIONS}

def fit_latency_model(C):
    """t_prefill = a + b*new_tokens + c*cached_tokens ; t_decode = d + e*n_out + f*n_out*context (fit on val profile)."""
    rows = C.LATROWS["val"] or C.LATROWS["test"]
    if not rows:                                  # stage 0: no latency profile yet
        C.LMOD = dict(cp=None, cd=None, r2_prefill=np.nan, r2_decode=np.nan); C.L_REF = np.nan
        for s in SPL:
            nanm = np.full_like(C.cache[s]["NIN"], np.nan)
            C.cache[s].update(lat_static=nanm, lat_casc0=nanm, lat_casc1=nanm)
        return
    Xp, yp, Xd, yd = [], [], [], []
    for r in rows:
        for k in K_OPTIONS:
            n = r.get(f"n_in_{k}")
            if r.get(f"t_pre_full_{k}") is not None: Xp.append([1, n, 0]); yp.append(r[f"t_pre_full_{k}"])
            if k > 0 and r.get(f"t_pre_inc_{k}") is not None:
                l = r[f"lcp_{k}"]; Xp.append([1, n - l, l]); yp.append(r[f"t_pre_inc_{k}"])
            if r.get(f"t_dec_{k}") is not None:
                o = r[f"n_out_{k}"]; Xd.append([1, o, o * n / 1000.0]); yd.append(r[f"t_dec_{k}"])
    Xp, yp, Xd, yd = map(np.asarray, (Xp, yp, Xd, yd))
    cp = np.linalg.lstsq(Xp, yp, rcond=None)[0]; cd = np.linalg.lstsq(Xd, yd, rcond=None)[0]
    r2 = lambda X, y, c: 1 - ((y - X @ c) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    C.LMOD = dict(cp=cp, cd=cd, r2_prefill=float(r2(Xp, yp, cp)), r2_decode=float(r2(Xd, yd, cd)))
    for s in SPL:
        D = C.cache[s]; NIN, NOUT, inc = D["NIN"], D["NOUT"], D["inc"]
        pf = cp[0] + cp[1] * NIN
        pinc = pf.copy(); pinc[:, 1:] = cp[0] + cp[1] * inc[:, 1:] + cp[2] * (NIN[:, 1:] - inc[:, 1:])
        dec = cd[0] + cd[1] * NOUT + cd[2] * NOUT * NIN / 1000.0
        D["lat_static"] = pf + dec
        D["lat_casc0"] = np.cumsum(pinc, 1) + dec
        D["lat_casc1"] = np.full_like(pf, np.nan); acc = pf[:, 1].copy()
        for j in range(1, K):
            if j > 1: acc = acc + pinc[:, j]
            D["lat_casc1"][:, j] = acc + dec[:, j]
    C.L_REF = float(C.cache["train"]["lat_static"][:, -1].mean())

# ----- observation groups -----
def _ret_feats(C, s): return ((C.TAB[s]["X"] - C.MU) / C.SD)[:, [FNAMES.index(f) for f in RETN]]
def _qry_feats(C, s): return ((C.TAB[s]["X"] - C.MU) / C.SD)[:, [FNAMES.index(f) for f in QRY]]

def hid_pca(C):
    if "pca" in C.cache: return C.cache["pca"]
    H = C.PR["train"].get("hid_mid")
    if H is None: C.cache["pca"] = None; return None
    Z = H.reshape(-1, H.shape[-1]).astype(np.float64); mu = Z.mean(0)
    sub = Z[np.random.RandomState(0).choice(len(Z), min(len(Z), 6000), replace=False)] - mu
    _, _, Vt = np.linalg.svd(sub, full_matrices=False); C.cache["pca"] = (mu, Vt[:HID_DIM].T); return C.cache["pca"]

def raw_obs(C, s, j, groups, start=0, pca=None):
    P, n = C.PR[s], len(C.TAB[s]["qids"]); parts = []
    mp = P["maxp"].astype(float)
    if "probe" in groups: parts.append(np.c_[mp[:, j], P["ent"][:, j], P["margin"][:, j]])
    if "agree" in groups:
        if j == 0: same, pprev, dmax = np.zeros(n), np.zeros(n), np.zeros(n)
        else:
            prev = P["top_ids"][:, j - 1, 0]; ids, pp = P["top_ids"][:, j], P["top_p"][:, j].astype(float)
            hit = ids == prev[:, None]; same = hit[:, 0].astype(float); pprev = (hit * pp).sum(1)
            dmax = mp[:, j] - mp[:, j - 1]
        parts.append(np.c_[same, pprev, dmax, mp[:, :j + 1].max(1), mp[:, 0]])
    if "hid" in groups:
        mu, W = pca if pca is not None else hid_pca(C)
        parts.append((P["hid_mid"][:, j].astype(np.float64) - mu) @ W)
    if "ret" in groups:
        sc = C.RS[s]; nxt = sc[:, j] if j < K - 1 else np.zeros(n); cur = sc[:, j - 1] if j > 0 else np.zeros(n)
        parts.append(np.c_[_ret_feats(C, s), nxt, cur])
    if "rr" in groups:
        rr = C.RR[s]
        if rr is None: raise ValueError("reranker scores missing; run stage B of the probe notebook")
        nxt = rr[:, j] if j < K - 1 else np.zeros(n); cur = rr[:, j - 1] if j > 0 else np.zeros(n)
        parts.append(np.c_[rr, nxt, cur])
    if "query" in groups: parts.append(_qry_feats(C, s))
    proc = C.cache[s]["proc0" if start == 0 else "proc1"][:, j]
    parts.append(np.c_[np.eye(K)[np.full(n, j)], (proc - C.cache[s]["NIN"][:, 0]) / C.T_REF])
    return np.concatenate(parts, 1)

def obs_tensor(C, s, groups, start=0, norm=None, pca=None):
    """(K, n, d) standardised observations; depths below `start` are filled but never used."""
    R = np.stack([raw_obs(C, s, j, groups, start, pca) if j >= start else raw_obs(C, s, start, groups, start, pca)
                  for j in range(K)], 0)
    if norm is None: return R.astype(np.float32)
    mu, sd = norm; return ((R - mu) / sd).astype(np.float32)

def obs_norm(C, groups, start=0, pca=None):
    R = obs_tensor(C, "train", groups, start, None, pca)[start:]
    Z = R.reshape(-1, R.shape[-1]); return Z.mean(0), Z.std(0) + 1e-6

def cascade_data(C, groups, lam, start=0, cost="tok", reuse=True, norm=None, pca=None):
    norm = norm or obs_norm(C, groups, start, pca)
    out = {}
    for s in SPL:
        D = C.cache[s]
        if cost == "tok":
            proc = D["proc0" if start == 0 else "proc1"] if reuse else D["proc0_noreuse"]
            c = (proc - D["NIN"][:, :1]) / C.T_REF
        else:
            lat = D["lat_casc0" if start == 0 else "lat_casc1"]; c = (lat - D["lat_static"][:, :1]) / C.L_REF
        out[s] = dict(O=obs_tensor(C, s, groups, start, norm, pca), Q=D["Q"], cost=np.nan_to_num(c), lam=lam, start=start)
    out["norm"] = norm; return out

## Evaluation (one accounting for every method) and frontier references

In [ ]:
def evaluate(C, s, k, kind, n_dec=1, pol_ms=0.0, extra_ms=0.0, trig=None, start=0, reuse=True):
    """Per-question metrics of a decision vector k (depth used for the answer).
    kind: static | oneshot (retrieval upfront, one generation) | cascade | jump | draft_up | draft | flare."""
    D, T = C.cache[s], C.TAB[s]; k = np.asarray(k, int); n = len(k); ar = np.arange(n)
    if kind == "flare":
        FL = T["FL"]; t = trig.astype(bool)
        f1 = np.where(t, FL["F1"], T["F1"][:, 0]); em = np.where(t, FL["EM"], T["EM"][:, 0])
        proc = D["NIN"][:, 0] + t * FL["NIN"]; outt = D["NOUT"][:, 0] + t * FL["NOUT"]
        cost_tok = np.where(t, FL["NIN"] - D["NIN"][:, 0], 0.0)
    else:
        f1, em = T["F1"][ar, k], T["EM"][ar, k]
        if kind in ("static", "oneshot"): proc = D["NIN"][ar, k]; outt = D["NOUT"][ar, k]
        elif kind == "cascade":
            P = D["proc0" if start == 0 else "proc1"] if reuse else D["proc0_noreuse"]
            proc = P[ar, k]; outt = D["NOUT"][ar, k]
        elif kind == "jump": proc = D["proc_jump"][ar, k]; outt = D["NOUT"][ar, k]
        elif kind in ("draft", "draft_up"):
            proc = D["NIN"][:, 0] + (k > 0) * D["NIN"][ar, k]; outt = D["NOUT"][:, 0] + (k > 0) * D["NOUT"][ar, k]
        cost_tok = proc - D["NIN"][:, 0]
        if kind in ("draft", "draft_up"): cost_tok = D["NIN"][ar, k] - D["NIN"][:, 0]      # paper convention
    ev = dict(f1=f1, em=em, k=k.astype(float), proc=proc.astype(float), total_tokens=(proc + outt).astype(float),
              reward=f1 + W_EM * em - LAMBDA_MAIN * cost_tok / C.T_REF)
    if s == "test":
        L = C.L; pf, pinc, dec, ret = L["pf"], L["pinc"], L["dec"], L["ret"]
        if kind == "static": lat = (k > 0) * ret + pf[ar, k] + dec[ar, k]
        elif kind == "oneshot": lat = ret + pol_ms + extra_ms + pf[ar, k] + dec[ar, k]
        elif kind == "cascade":
            src = pinc if reuse else pf
            cum = np.cumsum(src, 1) if start == 0 else np.c_[np.zeros(n), pf[:, 1], pf[:, 1] + np.cumsum(src[:, 2:], 1)]
            lat = ret + extra_ms + n_dec * pol_ms + cum[ar, k] + dec[ar, k]
        elif kind == "jump": lat = ret + pol_ms + pf[:, 0] + np.where(k == 0, 0.0, L["pjump"][ar, k]) + dec[ar, k]
        elif kind in ("draft", "draft_up"):
            lat = pf[:, 0] + dec[:, 0] + pol_ms + ((k > 0) | (kind == "draft_up")) * ret + (k > 0) * (pf[ar, k] + dec[ar, k])
        elif kind == "flare":
            lat = pf[:, 0] + dec[:, 0] + t * (L["t_flare_embed"] + L["t_flare_search"] + L["t_flare_pre"] + L["t_flare_dec"])
        ev["latency_ms"] = np.where(L["prof"], lat, np.nan)
    return ev

def summ(ev):
    return {m: float(np.nanmean(ev[m])) for m in ["f1", "em", "k", "proc", "total_tokens", "reward", "latency_ms"] if m in ev}

def seed_avg(evs):
    """Per-question average over seeds (dict of arrays)."""
    return {m: np.nanmean(np.stack([e[m] for e in evs]), 0) for m in evs[0] if isinstance(evs[0][m], np.ndarray)}

# ----- static frontier (upper concave hull of the four fixed depths) and matched mixtures -----
def static_evs(C, s="test"): return [evaluate(C, s, np.full(len(C.TAB[s]["qids"]), k), "static") for k in K_OPTIONS]

def hull(xs, ys):
    pts = sorted(zip(xs, ys, range(len(xs)))); h = []
    for p in pts:
        if h and p[1] <= h[-1][1]: continue
        while len(h) >= 2:
            (x1, y1, _), (x2, y2, _) = h[-2], h[-1]
            if (y2 - y1) * (p[0] - x1) <= (p[1] - y1) * (x2 - x1): h.pop()
            else: break
        h.append(p)
    return h

def matched_mix(C, cost_metric, target, metric="f1", s="test", ref_split=None):
    """Best mixture of two fixed depths whose mean cost equals `target`; per-question expectation.
    ref_split=None: hull and weights from the evaluation split (strongest, oracle-like reference);
    ref_split='val': hull and weights from validation (deployable reference)."""
    E = static_evs(C, s); R = static_evs(C, ref_split) if ref_split else E
    cm = cost_metric if (ref_split is None or cost_metric != "latency_ms") else "total_tokens"
    xs = [np.nanmean(e[cm]) for e in R]; ys = [np.nanmean(e[metric]) for e in R]
    h = hull(xs, ys); hx = [p[0] for p in h]; hi = [p[2] for p in h]
    if target <= hx[0]: w = {hi[0]: 1.0}
    elif target >= hx[-1]: w = {hi[-1]: 1.0}
    else:
        b = next(i for i in range(len(hx)) if hx[i] >= target); a = b - 1
        p = (target - hx[a]) / (hx[b] - hx[a]); w = {hi[a]: 1 - p, hi[b]: p}
    out = {m: sum(wt * E[i][m] for i, wt in w.items()) for m in E[0] if isinstance(E[0][m], np.ndarray)}
    out["mix"] = {K_OPTIONS[i]: round(wt, 3) for i, wt in w.items()}; return out

def vs_frontier(C, ev, cost_metric="total_tokens", metric="f1"):
    """Paired comparison with the static mixture at the method's own mean cost (latency: profiled questions)."""
    m = np.isfinite(ev[cost_metric]) if cost_metric == "latency_ms" else np.ones(len(ev[metric]), bool)
    if not m.any(): return np.nan, np.nan, None                                       # no latency profile yet
    mix = matched_mix(C, cost_metric, float(np.nanmean(ev[cost_metric][m])), metric)
    d = ev[metric][m] - mix[metric][m]
    return float(d.mean()), wilcoxon_p(ev[metric][m], mix[metric][m]), mix["mix"]

def boot_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; r = np.random.RandomState(seed)
    return np.percentile(x[r.randint(len(x), size=(n, len(x)))].mean(1), [2.5, 97.5])
def wilcoxon_p(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); d = d[np.isfinite(d)]
    if not np.any(d != 0): return 1.0
    try: return float(sps.wilcoxon(d, zero_method="zsplit").pvalue)
    except ValueError: return 1.0
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(o): run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj

## Learners

In [ ]:
class MLP(nn.Module):
    def __init__(self, d, out, hidden=POLICY_HIDDEN):
        super().__init__(); L, h = [], d
        for w in hidden: L += [nn.Linear(h, w), nn.ReLU()]; h = w
        L.append(nn.Linear(h, out)); self.net = nn.Sequential(*L)
    def forward(self, x): return self.net(x)

def pol_ms(net, d):
    x = torch.zeros(1, d); ts = []
    with torch.no_grad():
        for _ in range(20): net(x)
        for _ in range(200):
            t = time.perf_counter(); net(x); ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))

# ----- cascade: greedy rollout and returns -----
def cascade_logits(actor, O):
    with torch.no_grad(): return torch.stack([actor(torch.as_tensor(O[j])) for j in range(K)], 0).numpy()   # (K,n,2)

def stop_depth_from_scores(stop_pref, start):
    """stop_pref: (K, n) boolean 'prefers STOP'; first depth >= start that stops (forced at K-1)."""
    sp = stop_pref.copy(); sp[:start] = False; sp[K - 1] = True
    return sp.argmax(0)

def greedy_cascade(actor, O, start):
    lg = cascade_logits(actor, O); return stop_depth_from_scores(lg[..., STOP] >= lg[..., NEXT], start)

def cascade_return(Dsplit, stop):
    ar = np.arange(len(stop)); return float((Dsplit["Q"][ar, stop] - Dsplit["lam"] * Dsplit["cost"][ar, stop]).mean())

def _rollout(actor, critic, Dt, idx, start, g):
    """Sample episodes for questions idx. Returns flat tensors of valid transitions and GAE inputs."""
    O, Q, Cst, lam = Dt["O"], Dt["Q"], Dt["cost"], Dt["lam"]
    E = len(idx); j = torch.full((E,), start, dtype=torch.long); alive = torch.ones(E, dtype=torch.bool)
    S, A, LP, R, DN, V, VAL, J = [], [], [], [], [], [], [], []
    for t in range(K - start):
        o = O[j, idx]
        with torch.no_grad():
            lg = _masked_logits(actor, o, j == K - 1)
            a = torch.multinomial(torch.softmax(lg, -1), 1, generator=g).squeeze(1)
            lp = torch.log_softmax(lg, -1).gather(1, a[:, None]).squeeze(1); v = critic(o).squeeze(1)
        jn = torch.clamp(j + 1, max=K - 1)
        r = torch.where(a == STOP, Q[idx, j], -lam * (Cst[idx, jn] - Cst[idx, j]))
        dn = (a == STOP)
        S.append(o); A.append(a); LP.append(lp); R.append(r); DN.append(dn); V.append(v); VAL.append(alive.clone()); J.append(j.clone())
        alive = alive & ~dn; j = torch.where(a == NEXT, jn, j)
        if not alive.any(): break
    return [torch.stack(x) for x in (S, A, LP, R, DN, V, VAL, J)]

def _gae(R, DN, V, gam_lam):
    T = R.shape[0]; adv = torch.zeros_like(R); nxt_a = torch.zeros_like(R[0]); nxt_v = torch.zeros_like(R[0])
    for t in reversed(range(T)):
        nd = (~DN[t]).float()
        delta = R[t] + nd * nxt_v - V[t]
        nxt_a = delta + gam_lam * nd * nxt_a; adv[t] = nxt_a; nxt_v = V[t]
    return adv, adv + V

def _masked_logits(actor, s, last):
    lg = actor(s); return torch.where(torch.stack([torch.zeros_like(last), last], 1), torch.full_like(lg, -1e9), lg)

def _tensors(Dsplit):
    return dict(O=torch.as_tensor(Dsplit["O"]), Q=torch.as_tensor(Dsplit["Q"], dtype=torch.float32),
                cost=torch.as_tensor(Dsplit["cost"], dtype=torch.float32), lam=float(Dsplit["lam"]))

def train_cascade_ppo(D, seed, episodes=None, epochs=None, rollout_eps=None, lr=None):
    c = CPPO; episodes = episodes or c["episodes"]; epochs = c["epochs"] if epochs is None else epochs
    ro = int(min(rollout_eps or c["rollout_eps"], max(32, episodes // 20))); n_upd = max(1, episodes // ro)
    ev_every = max(1, n_upd // 25); start = D["train"]["start"]
    set_seed(seed); g = torch.Generator().manual_seed(seed); Dt = _tensors(D["train"]); n = Dt["Q"].shape[0]
    d = Dt["O"].shape[-1]; actor, critic = MLP(d, 2), MLP(d, 1)
    params = list(actor.parameters()) + list(critic.parameters()); opt = torch.optim.Adam(params, lr=lr or c["lr"])
    best, best_state, log = -1e9, copy.deepcopy(actor.state_dict()), []
    for u in range(n_upd):
        idx = torch.randint(n, (ro,), generator=g)
        S, A, LP, R, DN, V, VAL, J = _rollout(actor, critic, Dt, idx, start, g)
        adv, ret = _gae(R, DN, V, c["gae"])
        m = VAL.flatten(); s_ = S.flatten(0, 1)[m]; a_ = A.flatten()[m]; lp_ = LP.flatten()[m]
        adv_ = adv.flatten()[m]; ret_ = ret.flatten()[m]; adv_ = (adv_ - adv_.mean()) / (adv_.std() + 1e-8)
        last_ = J.flatten()[m] == K - 1
        mb = max(64, min(c["minibatch"], len(a_) // 4))
        for _ in range(epochs):
            for b in torch.randperm(len(a_), generator=g).split(mb):
                lg = _masked_logits(actor, s_[b], last_[b]); lsm = torch.log_softmax(lg, -1)
                ratio = torch.exp(lsm.gather(1, a_[b, None]).squeeze(1) - lp_[b])
                pg = -torch.min(ratio * adv_[b], torch.clamp(ratio, 1 - c["clip"], 1 + c["clip"]) * adv_[b]).mean()
                p = lsm.exp(); ent = -(p * lsm.clamp(min=-30)).sum(1).mean()
                vl = ((critic(s_[b]).squeeze(1) - ret_[b]) ** 2).mean()
                loss = pg + c["vf"] * vl - c["ent"] * ent
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(params, 0.5); opt.step()
        if (u + 1) % ev_every == 0 or u == n_upd - 1:
            v = cascade_return(D["val"], greedy_cascade(actor, D["val"]["O"], start))
            log.append(dict(episodes=(u + 1) * ro, val_return=v))
            if v > best: best, best_state = v, copy.deepcopy(actor.state_dict())
    actor.load_state_dict(best_state); return actor, log

def train_cascade_reinforce(D, seed, episodes=None, batch_eps=None, lr=None):
    c = CRF; episodes = episodes or c["episodes"]; be = int(min(batch_eps or c["batch_eps"], max(32, episodes // 20)))
    n_upd = max(1, episodes // be); ev_every = max(1, n_upd // 25); start = D["train"]["start"]
    set_seed(seed); g = torch.Generator().manual_seed(seed); Dt = _tensors(D["train"]); n = Dt["Q"].shape[0]
    d = Dt["O"].shape[-1]; actor, critic = MLP(d, 2), MLP(d, 1)
    params = list(actor.parameters()) + list(critic.parameters()); opt = torch.optim.Adam(params, lr=lr or c["lr"])
    best, best_state, log = -1e9, copy.deepcopy(actor.state_dict()), []
    for u in range(n_upd):
        idx = torch.randint(n, (be,), generator=g)
        S, A, LP, R, DN, V, VAL, J = _rollout(actor, critic, Dt, idx, start, g)
        G = torch.flip(torch.cumsum(torch.flip(R * VAL.float(), [0]), 0), [0])        # gamma = 1 returns
        m = VAL.flatten(); s_ = S.flatten(0, 1)[m]; a_ = A.flatten()[m]; G_ = G.flatten()[m]
        last_ = J.flatten()[m] == K - 1
        lsm = torch.log_softmax(_masked_logits(actor, s_, last_), -1); v = critic(s_).squeeze(1)
        adv = (G_ - v).detach(); p = lsm.exp()
        loss = -(lsm.gather(1, a_[:, None]).squeeze(1) * adv).mean() + c["vf"] * ((v - G_) ** 2).mean() \
               - c["ent"] * (-(p * lsm.clamp(min=-30)).sum(1).mean())
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(params, 0.5); opt.step()
        if (u + 1) % ev_every == 0 or u == n_upd - 1:
            vr = cascade_return(D["val"], greedy_cascade(actor, D["val"]["O"], start))
            log.append(dict(episodes=(u + 1) * be, val_return=vr))
            if vr > best: best, best_state = vr, copy.deepcopy(actor.state_dict())
    actor.load_state_dict(best_state); return actor, log

def train_cascade_fullinfo(D, seed):
    """Not deployable online: maximises the exact expected return using the rewards of every stopping depth."""
    c = CFI; start = D["train"]["start"]; set_seed(seed); g = torch.Generator().manual_seed(seed)
    Dt = _tensors(D["train"]); n = Dt["Q"].shape[0]; d = Dt["O"].shape[-1]; actor = MLP(d, 2)
    Rst = Dt["Q"] - Dt["lam"] * Dt["cost"]
    opt = torch.optim.Adam(actor.parameters(), lr=c["lr"], weight_decay=c["wd"])
    best, best_state, wait = -1e9, copy.deepcopy(actor.state_dict()), 0
    for ep in range(c["max_epochs"]):
        for b in torch.randperm(n, generator=g).split(c["batch"]):
            lg = torch.stack([actor(Dt["O"][j, b]) for j in range(K)], 0)               # (K,B,2)
            ps = torch.softmax(lg, -1)[..., STOP]
            ps = torch.cat([torch.zeros_like(ps[:start]), ps[start:K - 1], torch.ones_like(ps[:1])], 0)
            surv = torch.cumprod(torch.cat([torch.ones_like(ps[:1]), 1 - ps[:-1]], 0), 0)
            pj = surv * ps; er = (pj * Rst[b].T).sum(0)
            hb = -(ps * torch.log(ps + 1e-8) + (1 - ps) * torch.log(1 - ps + 1e-8))
            loss = -er.mean() - c["ent"] * (surv * hb).sum(0).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        v = cascade_return(D["val"], greedy_cascade(actor, D["val"]["O"], start))
        if v > best + 1e-7: best, best_state, wait = v, copy.deepcopy(actor.state_dict()), 0
        else:
            wait += 1
            if wait >= c["patience"]: break
    actor.load_state_dict(best_state); return actor, []

def threshold_cascade(C, D, score="maxp", start=0):
    """Training-free rule tuned on validation: stop at the first depth whose score >= tau_j."""
    def sc(s):
        P = C.PR[s]; mp = P["maxp"].astype(float)
        if score == "maxp": return mp.T
        out = mp.copy()
        for j in range(1, K):
            hit = P["top_ids"][:, j] == P["top_ids"][:, j - 1, :1]; out[:, j] = (hit * P["top_p"][:, j].astype(float)).sum(1)
        return out.T
    Sv = sc("val"); grid = np.unique(np.r_[np.quantile(Sv[start:K - 1].ravel(), np.linspace(0, 1, 41)), 1.01])
    def stops(S, tau): return stop_depth_from_scores(S >= np.asarray(tau)[:, None], start)
    best = max(((cascade_return(D["val"], stops(Sv, [t] * K)), [t] * K) for t in grid), key=lambda x: x[0])
    tau = list(best[1])
    for _ in range(2):                                                                # coordinate ascent per depth
        for j in range(start, K - 1):
            cand = []
            for t in grid:
                tt = tau.copy(); tt[j] = t; cand.append((cascade_return(D["val"], stops(Sv, tt)), t))
            tau[j] = max(cand)[1]
    return dict(tau=tau, stop_fn=lambda s: stops(sc(s), tau))

# ----- one-shot (bandit) learners, identical to the paper's RL-AR -----
def train_bandit_ppo(Xtr, Rtr, Xva, Rva, seed, total=None, eval_every=2):
    c = BPPO; total = total or c["total_samples"]; ro = min(c["rollout"], max(32, total // 20)); n_upd = max(1, total // ro)
    set_seed(seed); g = torch.Generator().manual_seed(seed)
    X, R = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Rtr, dtype=torch.float32)
    actor, critic = MLP(X.shape[1], R.shape[1]), MLP(X.shape[1], 1)
    params = list(actor.parameters()) + list(critic.parameters()); opt = torch.optim.Adam(params, lr=c["lr"])
    Xv = torch.tensor(Xva, dtype=torch.float32); best, best_state = -1e9, copy.deepcopy(actor.state_dict())
    for u in range(n_upd):
        idx = torch.randint(len(X), (ro,), generator=g); s = X[idx]
        with torch.no_grad():
            pr = torch.softmax(actor(s), -1); a = torch.multinomial(pr, 1, generator=g).squeeze(1)
            lpo = torch.log(pr.gather(1, a[:, None]).squeeze(1) + 1e-12); r = R[idx, a]
            adv = r - critic(s).squeeze(1); adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        for _ in range(c["epochs"]):
            for b in torch.randperm(ro, generator=g).split(min(c["minibatch"], ro)):
                lp = torch.log_softmax(actor(s[b]), -1); ratio = torch.exp(lp.gather(1, a[b, None]).squeeze(1) - lpo[b])
                pg = -torch.min(ratio * adv[b], torch.clamp(ratio, 1 - c["clip"], 1 + c["clip"]) * adv[b]).mean()
                loss = pg + c["vf"] * ((critic(s[b]).squeeze(1) - r[b]) ** 2).mean() - c["ent"] * (-(lp.exp() * lp).sum(-1).mean())
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(params, 0.5); opt.step()
        if (u + 1) % eval_every == 0 or u == n_upd - 1:
            with torch.no_grad(): v = float(Rva[np.arange(len(Rva)), actor(Xv).argmax(1).numpy()].mean())
            if v > best: best, best_state = v, copy.deepcopy(actor.state_dict())
    actor.load_state_dict(best_state); return actor

def train_reward_regression(Xtr, Rtr, Xva, Rva, seed):
    """Full-information, cost-aware supervised baseline: regress r(q,k) for every k and act greedily."""
    set_seed(seed); g = torch.Generator().manual_seed(seed)
    X, T = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Rtr, dtype=torch.float32); Xv = torch.tensor(Xva, dtype=torch.float32)
    net = MLP(X.shape[1], T.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=CFI["lr"], weight_decay=CFI["wd"])
    best, best_state, wait = -1e9, copy.deepcopy(net.state_dict()), 0
    for ep in range(CFI["max_epochs"]):
        for b in torch.randperm(len(X), generator=g).split(CFI["batch"]):
            loss = ((net(X[b]) - T[b]) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): v = float(Rva[np.arange(len(Rva)), net(Xv).argmax(1).numpy()].mean())
        if v > best + 1e-7: best, best_state, wait = v, copy.deepcopy(net.state_dict()), 0
        else:
            wait += 1
            if wait >= CFI["patience"]: break
    net.load_state_dict(best_state); return net

def act(net, X):
    with torch.no_grad(): return net(torch.as_tensor(X, dtype=torch.float32)).argmax(1).numpy()

print("Learners ready.")

## Method registry
Every method returns per-seed test evaluations (and the validation decisions needed for budget calibration).

In [ ]:
BASE = ("probe", "agree", "ret", "query")
FEATURE_SETS = {"base": BASE, "base+hid": BASE + ("hid",), "base+rr": BASE + ("rr",), "base+hid+rr": BASE + ("hid", "rr")}
LITE, LITE_RR = ("ret", "query"), ("ret", "query", "rr")

def oneshot_X(C, s, groups, j0probe=False):
    parts = []
    if "ret" in groups: parts.append(_ret_feats(C, s))
    if "query" in groups: parts.append(_qry_feats(C, s))
    if "rr" in groups: parts.append(C.RR[s])
    if j0probe:
        P = C.PR[s]; parts.append(np.c_[P["maxp"][:, 0], P["ent"][:, 0], P["margin"][:, 0]])
    return np.concatenate(parts, 1).astype(np.float32)

def oneshot_reward(C, s, lam, kind):
    D = C.cache[s]; proc = D["proc_jump"] if kind == "jump" else D["NIN"]
    return D["Q"] - lam * (proc - D["NIN"][:, :1]) / C.T_REF

def run_oneshot(C, name, groups, lam, seeds, learner="ppo", kind="oneshot", j0probe=False):
    def go():
        Xs = {s: oneshot_X(C, s, groups, j0probe) for s in SPL}; mu, sd = Xs["train"].mean(0), Xs["train"].std(0) + 1e-6
        Xs = {s: ((x - mu) / sd).astype(np.float32) for s, x in Xs.items()}
        Rtr, Rva = oneshot_reward(C, "train", lam, kind), oneshot_reward(C, "val", lam, kind)
        out = {}
        for sd_ in seeds:
            net = train_bandit_ppo(Xs["train"], Rtr, Xs["val"], Rva, sd_) if learner == "ppo" else \
                  train_reward_regression(Xs["train"], Rtr, Xs["val"], Rva, sd_)
            pm = pol_ms(net, Xs["test"].shape[1]); extra = C.RR_MS if "rr" in groups else 0.0
            out[sd_] = dict(test=evaluate(C, "test", act(net, Xs["test"]), kind, pol_ms=pm, extra_ms=extra),
                            val_k=act(net, Xs["val"]))
        return out
    return cached(f"{C.out}/oneshot_{name}_{tag(lam)}_n{len(seeds)}.pkl", go)

def run_cascade(C, name, groups, lam, seeds, learner="ppo", start=0, cost="tok", reuse=True, episodes=None, epochs=None):
    def go():
        D = cascade_data(C, groups, lam if cost == "tok" else lam, start, cost, reuse)
        out = {}
        for sd_ in seeds:
            if learner == "ppo": actor, log = train_cascade_ppo(D, sd_, episodes=episodes, epochs=epochs)
            elif learner == "reinforce": actor, log = train_cascade_reinforce(D, sd_, episodes=episodes)
            else: actor, log = train_cascade_fullinfo(D, sd_)
            st = greedy_cascade(actor, D["test"]["O"], start); pm = pol_ms(actor, D["test"]["O"].shape[-1])
            extra = C.RR_MS if "rr" in groups else 0.0
            out[sd_] = dict(test=evaluate(C, "test", st, "cascade", n_dec=st - start + 1, pol_ms=pm, extra_ms=extra,
                                          start=start, reuse=reuse),
                            val_k=greedy_cascade(actor, D["val"]["O"], start), log=log,
                            val_return=cascade_return(D["val"], greedy_cascade(actor, D["val"]["O"], start)))
        return out
    tg = f"{name}_{learner}_{'-'.join(groups)}_s{start}_{cost}_{'r' if reuse else 'nr'}_{tag(lam)}_{episodes or 'def'}_{epochs if epochs is not None else 'def'}_n{len(seeds)}"
    return cached(f"{C.out}/casc_{tg}.pkl", go)

def run_threshold(C, score, lam, start=0):
    def go():
        D = cascade_data(C, ("probe",), lam, start)
        tc = threshold_cascade(C, D, score, start); st = tc["stop_fn"]("test")
        return {0: dict(test=evaluate(C, "test", st, "cascade", n_dec=st - start + 1, start=start),
                        val_k=tc["stop_fn"]("val"), tau=tc["tau"])}
    return cached(f"{C.out}/tc_{score}_s{start}_{tag(lam)}.pkl", go)

def run_paper_baselines(C):
    """Static depths, Adaptive-k, the paper's uncertainty threshold and FLARE rules, and the paper's RL-AR runs."""
    def go():
        T = {s: C.TAB[s] for s in SPL}; n = len(T["test"]["qids"]); out = {}
        for k in K_OPTIONS: out[f"Static k={k}"] = {0: dict(test=evaluate(C, "test", np.full(n, k), "static"))}
        kb = int(T["val"]["F1"].mean(0).argmax()); out["Static k* (val F1)"] = {0: dict(test=evaluate(C, "test", np.full(n, kb), "static"))}
        gap = -np.diff(C.RS["test"][:, :K + 0], axis=1)                          # gaps after ranks 1, 2, 3 (Taguchi et al.)
        out["Adaptive-k (similarity gap)"] = {0: dict(test=evaluate(C, "test", gap.argmax(1) + 1, "oneshot"))}
        conf = lambda s: T[s]["X"][:, FNAMES.index("first_maxp")]
        Rv = C.cache["val"]["Q"] - LAMBDA_MAIN * (C.cache["val"]["NIN"] - C.cache["val"]["NIN"][:, :1]) / C.T_REF
        best = (-1e9, None, None)
        for tau in np.unique(np.quantile(conf("val"), np.linspace(0, 1, 41))):
            for kk in range(1, K):
                ki = np.where(conf("val") >= tau, 0, kk); v = Rv[np.arange(len(ki)), ki].mean()
                if v > best[0]: best = (v, tau, kk)
        out["Uncertainty threshold"] = {0: dict(test=evaluate(C, "test", np.where(conf("test") >= best[1], 0, best[2]), "draft"))}
        fbest = (-1e9, None); FLv = T["val"]["FL"]
        for th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            ev = evaluate(C, "val", np.zeros(len(FLv["F1"]), int), "flare", trig=FLv["MINP"] < th); v = ev["reward"].mean()
            if v > fbest[0]: fbest = (v, th)
        trig = T["test"]["FL"]["MINP"] < fbest[1]
        out["FLARE"] = {0: dict(test=evaluate(C, "test", np.where(trig, 2, 0), "flare", trig=trig))}
        if "main_ppo" in C.OLDRES:
            out["RL-AR (draft, paper)"] = {sd: dict(test=evaluate(C, "test", k, "draft_up")) for sd, k in C.OLDRES["main_ppo"].items()}
        if "main_ppo_nodraft" in C.OLDRES:
            out["RL-AR-Lite (paper)"] = {sd: dict(test=evaluate(C, "test", k, "oneshot")) for sd, k in C.OLDRES["main_ppo_nodraft"].items()}
        orc = (C.cache["test"]["Q"] - LAMBDA_MAIN * (C.cache["test"]["NIN"] - C.cache["test"]["NIN"][:, :1]) / C.T_REF).argmax(1)
        out["Oracle depth (upper bound)"] = {0: dict(test=evaluate(C, "test", orc, "static"))}
        return out
    return cached(f"{C.out}/paper_baselines.pkl", go)

def table_row(C, name, res):
    evs = [r["test"] for r in res.values()]; S = pd.DataFrame([summ(e) for e in evs]); avg = seed_avg(evs)
    dt, pt, mixt = vs_frontier(C, avg, "total_tokens"); dl, pl, mixl = vs_frontier(C, avg, "latency_ms")
    row = dict(ds=C.ds, model=C.mk, method=name, seeds=len(evs), **S.mean().to_dict(), f1_sd=float(S["f1"].std(ddof=1)) if len(evs) > 1 else 0.0,
               dF1_vs_mix_tok=dt, p_tok=pt, dF1_vs_mix_lat=dl, p_lat=pl, mix_tok=str(mixt), mix_lat=str(mixl))
    lo, hi = boot_ci(avg["f1"]); row.update(f1_lo=lo, f1_hi=hi); return row

def build_cfg():
    out = {}
    for ds, mk in configs():
        C = load_cfg(ds, mk)
        if C is None: continue
        out[(ds, mk)] = C
        print(f"{DNAME[ds]:8s} {MNAME[mk]:16s} n_in match {C.nin_match} | decode-answer match {C.match_rate} | "
              f"latency model R2 prefill {C.LMOD['r2_prefill']:.3f} decode {C.LMOD['r2_decode']:.3f}")
    return out
CFG = build_cfg(); print(len(CFG), "configs with a probe cache")

## G0. Is the probe informative, and does the cascade beat the static frontier? (gate)

In [ ]:
G0_ROWS, AUC_ROWS = [], []
if RUN["G0"]:
    for (ds, mk), C in CFG.items():
        for s in ["val", "test"]:
            for k in K_OPTIONS:
                y = C.TAB[s]["F1"][:, k] >= 0.5
                if 0 < y.mean() < 1:
                    AUC_ROWS.append(dict(ds=ds, model=mk, split=s, k=k, acc=float(y.mean()),
                                         auroc_maxp=roc_auc_score(y, C.PR[s]["maxp"][:, k]),
                                         auroc_ret_s1=roc_auc_score(y, C.RS[s][:, 0])))
        fi = run_cascade(C, "g0", BASE, LAMBDA_MAIN, [0], learner="fullinfo")
        tc = run_threshold(C, "maxp", LAMBDA_MAIN)
        for nm, res in [("Cascade (full information)", fi), ("Threshold cascade (maxp)", tc)]:
            r = table_row(C, nm, res); G0_ROWS.append(r)
    auc = pd.DataFrame(AUC_ROWS); g0 = pd.DataFrame(G0_ROWS)
    display(auc[auc.split == "test"].pivot_table(index=["ds", "model"], columns="k", values=["auroc_maxp", "auroc_ret_s1"]).round(3))
    display(g0[["ds", "model", "method", "f1", "total_tokens", "latency_ms", "dF1_vs_mix_tok", "p_tok", "dF1_vs_mix_lat", "p_lat"]].round(4))
    fi_rows = g0[g0.method == "Cascade (full information)"]
    med_tok = float(fi_rows.dF1_vs_mix_tok.median()); n_lat = int((fi_rows.dF1_vs_mix_lat >= 0).sum())
    lat_known = bool(fi_rows.dF1_vs_mix_lat.notna().any())
    GO = (med_tok >= 0.010) and (not lat_known or n_lat >= int(math.ceil(len(fi_rows) * 10 / 15)))
    if not lat_known: print("Latency not profiled yet: stage-0 gate uses the token criterion only; the latency criterion is checked in stage 1.")
    print(f"\nG0 GATE: median dF1 vs static mixture at matched tokens = {med_tok:+.4f} (need >= +0.010); "
          f"configs not below the mixture at matched latency = {n_lat}/{len(fi_rows)} (need >= {math.ceil(len(fi_rows)*10/15)})")
    print("G0 decision:", "GO" if GO else "NO-GO (the probe signal is not strong enough; do not claim the cascade as the method)")
    g0.to_csv(f"{STUDY}/g0_gate.csv", index=False); auc.to_csv(f"{STUDY}/g0_auroc.csv", index=False)
else:
    GO = True
PROCEED = GO or IGNORE_GATE

In [ ]:
# ---------------- stage 1 (automatic after GO): every pair gets probes, latency and reranker scores ----------------
if RUN["G0"] and GO and AUTO_CONTINUE:
    t0 = time.time()
    for ds, mk in available_pairs(): probe_config(ds, mk, do_latency=True)
    for ds in sorted({d for d, _ in available_pairs()}): pc_rerank(ds)
    print(f"stage-1 GPU work: {time.time()-t0:.0f}s")
    RUN = {k: True for k in RUN}; CFG = build_cfg(); PROCEED = True
    print("Stage 1: running all blocks on", len(CFG), "configs")
elif not GO:
    print("Gate failed: stopping here. Send the G0 tables; no further GPU or CPU time is spent.")

## Feature-set selection (validation only) and P1: main comparison at the main cost weight

In [ ]:
if PROCEED and RUN["P1"]:
    sel = []
    for (ds, mk), C in CFG.items():
        for fs, groups in FEATURE_SETS.items():
            if "rr" in groups and C.RR["train"] is None: continue
            if "hid" in groups and "hid_mid" not in C.PR["train"]: continue
            res = run_cascade(C, f"sel_{fs}", groups, LAMBDA_MAIN, FRONT_SEEDS)
            sel.append(dict(ds=ds, model=mk, fs=fs, val_return=np.mean([r["val_return"] for r in res.values()])))
    sel = pd.DataFrame(sel); piv = sel.pivot_table(index=["ds", "model"], columns="fs", values="val_return")
    piv = piv.dropna(axis=1); MAIN_FS = piv.mean().idxmax(); MAIN_G = FEATURE_SETS[MAIN_FS]
    display(piv.round(4)); print("Pre-registered choice by mean validation return:", MAIN_FS, MAIN_G)
    json.dump(dict(main_feature_set=MAIN_FS, groups=list(MAIN_G)), open(f"{STUDY}/main_feature_set.json", "w"))

    ROWS = []
    for (ds, mk), C in CFG.items():
        M = dict(run_paper_baselines(C))
        if C.RR["train"] is not None:
            M["Lite+rerank (bandit PPO)"] = run_oneshot(C, "literr_ppo", LITE_RR, LAMBDA_MAIN, SEEDS)
            M["Lite+rerank (reward regression, full info)"] = run_oneshot(C, "literr_reg", LITE_RR, LAMBDA_MAIN, SEEDS, learner="reg")
        M["Probe-0 + jump (bandit PPO)"] = run_oneshot(C, "probe0jump", LITE, LAMBDA_MAIN, SEEDS, kind="jump", j0probe=True)
        M["Threshold cascade (maxp)"] = run_threshold(C, "maxp", LAMBDA_MAIN)
        M["Threshold cascade (agreement)"] = run_threshold(C, "agree", LAMBDA_MAIN)
        M["RL-AR-Cascade (PPO, ours)"] = run_cascade(C, "main", MAIN_G, LAMBDA_MAIN, SEEDS)
        M["Cascade (REINFORCE)"] = run_cascade(C, "main", MAIN_G, LAMBDA_MAIN, SEEDS, learner="reinforce")
        M["Cascade (full information)"] = run_cascade(C, "main", MAIN_G, LAMBDA_MAIN, SEEDS, learner="fullinfo")
        for nm, res in M.items(): ROWS.append(table_row(C, nm, res))
        C.MAIN = M
    P1 = pd.DataFrame(ROWS)
    for col in ["p_tok", "p_lat"]:
        P1[col + "_holm"] = np.nan
        for nm, g in P1.groupby("method"): P1.loc[g.index, col + "_holm"] = holm(g[col].values)
    P1.to_csv(f"{STUDY}/p1_main.csv", index=False)
    show = ["ds", "model", "method", "f1", "f1_sd", "k", "total_tokens", "latency_ms", "reward", "dF1_vs_mix_tok", "p_tok_holm", "dF1_vs_mix_lat", "p_lat_holm"]
    display(P1[show].round(4))
    agg = P1.groupby("method")[["f1", "total_tokens", "latency_ms", "dF1_vs_mix_tok", "dF1_vs_mix_lat"]].mean()
    agg["n_better_tok"] = P1.assign(b=(P1.dF1_vs_mix_tok > 0) & (P1.p_tok_holm < 0.05)).groupby("method").b.sum()
    agg["n_worse_tok"] = P1.assign(b=(P1.dF1_vs_mix_tok < 0) & (P1.p_tok_holm < 0.05)).groupby("method").b.sum()
    agg["n_better_lat"] = P1.assign(b=(P1.dF1_vs_mix_lat > 0) & (P1.p_lat_holm < 0.05)).groupby("method").b.sum()
    agg["n_worse_lat"] = P1.assign(b=(P1.dF1_vs_mix_lat < 0) & (P1.p_lat_holm < 0.05)).groupby("method").b.sum()
    display(agg.sort_values("dF1_vs_mix_tok", ascending=False).round(4)); agg.to_csv(f"{STUDY}/p1_summary.csv")

## P2. Quality-cost frontiers (tokens and latency)

In [ ]:
if PROCEED and "MAIN_G" not in globals() and os.path.exists(f"{STUDY}/main_feature_set.json"):
    _m = json.load(open(f"{STUDY}/main_feature_set.json")); MAIN_FS, MAIN_G = _m["main_feature_set"], tuple(_m["groups"])

if PROCEED and RUN["P2"]:
    FR = []
    for (ds, mk), C in CFG.items():
        for lam in LAMBDA_GRID:
            runs = {"RL-AR-Cascade (PPO, ours)": run_cascade(C, "front", MAIN_G, lam, FRONT_SEEDS, episodes=FRONT_EPISODES),
                    "Cascade (full information)": run_cascade(C, "front", MAIN_G, lam, FRONT_SEEDS, learner="fullinfo"),
                    "Probe-0 + jump (bandit PPO)": run_oneshot(C, "probe0jump", LITE, lam, FRONT_SEEDS, kind="jump", j0probe=True),
                    "RL-AR-Lite (bandit PPO)": run_oneshot(C, "lite_ppo", LITE, lam, FRONT_SEEDS)}
            if C.RR["train"] is not None:
                runs["Lite+rerank (reward regression, full info)"] = run_oneshot(C, "literr_reg", LITE_RR, lam, FRONT_SEEDS, learner="reg")
            runs["Threshold cascade (agreement)"] = run_threshold(C, "agree", lam)
            for nm, res in runs.items():
                FR.append(dict(lam=lam, **table_row(C, nm, res)))
        for k in K_OPTIONS:
            FR.append(dict(lam=np.nan, **table_row(C, f"Static k={k}", C.MAIN[f"Static k={k}"] if hasattr(C, "MAIN") else run_paper_baselines(C)[f"Static k={k}"])))
    FR = pd.DataFrame(FR); FR.to_csv(f"{STUDY}/p2_frontier.csv", index=False)
    inside = FR[FR.lam.notna()]
    area = inside.groupby(["method", "ds", "model"])[["dF1_vs_mix_tok", "dF1_vs_mix_lat"]].mean().groupby("method").agg(["mean", "min", "max"])
    print("Mean F1 above (+) or below (-) the static-mixture frontier along each learned curve:"); display(area.round(4))
    area.to_csv(f"{STUDY}/p2_area.csv")
    for cm, lab in [("total_tokens", "Total tokens per query"), ("latency_ms", "End-to-end latency (ms)")]:
        dss = [d for d in PS_DATASETS if any(k[0] == d for k in CFG)]
        fig, axes = plt.subplots(len(dss), len(PS_MODELS), figsize=(3.2 * len(PS_MODELS), 2.8 * len(dss)), squeeze=False)
        for r_, ds in enumerate(dss):
            for c_, mk in enumerate(PS_MODELS):
                ax = axes[r_, c_]
                if (ds, mk) not in CFG: ax.axis("off"); continue
                d = FR[(FR.ds == ds) & (FR.model == mk)]
                st = d[d.method.str.startswith("Static")].sort_values(cm)
                h = hull(list(st[cm]), list(st.f1)); ax.plot([p[0] for p in h], [p[1] for p in h], "k--", lw=1, label="Static frontier")
                ax.scatter(st[cm], st.f1, c="k", marker="D", s=14, zorder=5)
                for nm, mkr in [("RL-AR-Cascade (PPO, ours)", "o-"), ("Cascade (full information)", "x:"),
                                ("Threshold cascade (agreement)", "^-"), ("RL-AR-Lite (bandit PPO)", "s-"), ("Probe-0 + jump (bandit PPO)", "v-")]:
                    e = d[d.method == nm].sort_values(cm)
                    if len(e): ax.plot(e[cm], e.f1, mkr, ms=3, lw=1, label=nm)
                ax.set_title(f"{DNAME[ds]} | {MNAME[mk]}", fontsize=8); ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
                if r_ == len(dss) - 1: ax.set_xlabel(lab, fontsize=8)
                if c_ == 0: ax.set_ylabel("F1", fontsize=8)
        axes[0, 0].legend(fontsize=6); fig.tight_layout()
        fig.savefig(f"{STUDY}/fig_frontier_{'tokens' if cm == 'total_tokens' else 'latency'}.pdf"); plt.show()

## P3. Label-free budget calibration
For a token budget B (a fraction of the validation prompt tokens of Static k=3), pick the smallest cost
weight whose validation decisions stay within B. Only prompt lengths are used, never answers.

In [ ]:
if PROCEED and RUN["P3"]:
    BR = []
    for (ds, mk), C in CFG.items():
        Dv = C.cache["val"]; B3 = float(Dv["NIN"][:, -1].mean())
        for frac in BUDGET_FRACS:
            B = frac * B3
            for nm, fn, kind in [("RL-AR-Cascade (PPO, ours)", lambda lam: run_cascade(C, "front", MAIN_G, lam, FRONT_SEEDS, episodes=FRONT_EPISODES), "cascade"),
                                 ("RL-AR-Lite (bandit PPO)", lambda lam: run_oneshot(C, "lite_ppo", LITE, lam, FRONT_SEEDS), "oneshot")]:
                procv = lambda res: np.mean([Dv["proc0" if kind == "cascade" else "NIN"][np.arange(len(r["val_k"])), r["val_k"]].mean() for r in res.values()])
                lam_ok = next((lam for lam in sorted(LAMBDA_GRID) if procv(fn(lam)) <= B + 1e-9), None)
                if lam_ok is None: continue
                res = fn(lam_ok); avg = seed_avg([r["test"] for r in res.values()])
                mixv = matched_mix(C, "proc", B, "f1", "test", ref_split="val")          # deployable: weights from val
                BR.append(dict(ds=ds, model=mk, method=nm, frac=frac, lam=lam_ok, budget=B,
                               test_proc=float(avg["proc"].mean()), adherence=float(avg["proc"].mean() / B),
                               f1=float(avg["f1"].mean()), f1_mix_val=float(mixv["f1"].mean()),
                               dF1_vs_mix=float((avg["f1"] - mixv["f1"]).mean()), p=wilcoxon_p(avg["f1"], mixv["f1"])))
    BR = pd.DataFrame(BR); BR.to_csv(f"{STUDY}/p3_budget.csv", index=False)
    display(BR.groupby(["method", "frac"])[["adherence", "f1", "f1_mix_val", "dF1_vs_mix"]].mean().round(4))

## P4. Why PPO? Interaction budget, data reuse and seed variance (bandit feedback only)

In [ ]:
if PROCEED and RUN["P4"]:
    IB = []
    for (ds, mk), C in CFG.items():
        D = cascade_data(C, MAIN_G, LAMBDA_MAIN)
        ar = np.arange(len(D["test"]["Q"])); Rte = D["test"]["Q"] - LAMBDA_MAIN * D["test"]["cost"]
        stat = lambda s: C.cache[s]["Q"] - LAMBDA_MAIN * (C.cache[s]["NIN"] - C.cache[s]["NIN"][:, :1]) / C.T_REF
        kst = int(stat("val").mean(0).argmax()); lo, hi = stat("test")[:, kst].mean(), stat("test").max(1).mean()   # 0 = best fixed depth, 1 = oracle depth
        for B in INTERACTION_BUDGETS:
            for nm, kw in [("PPO (10 epochs)", dict(learner="ppo")), ("PPO (1 epoch, no reuse)", dict(learner="ppo", epochs=1)),
                           ("REINFORCE", dict(learner="reinforce"))]:
                res = run_cascade(C, "ib", MAIN_G, LAMBDA_MAIN, FRONT_SEEDS, episodes=B, **kw)
                for sd_, r in res.items():
                    st = r["test"]["k"].astype(int); v = Rte[ar, st].mean()
                    IB.append(dict(ds=ds, model=mk, budget=B, learner=nm, seed=sd_, norm_return=(v - lo) / (hi - lo + 1e-9)))
        fi = run_cascade(C, "main", MAIN_G, LAMBDA_MAIN, SEEDS, learner="fullinfo")
        for sd_, r in fi.items():
            st = r["test"]["k"].astype(int)
            IB.append(dict(ds=ds, model=mk, budget=np.nan, learner="Full information (reference)", seed=sd_, norm_return=(Rte[ar, st].mean() - lo) / (hi - lo + 1e-9)))
    IB = pd.DataFrame(IB); IB.to_csv(f"{STUDY}/p4_interaction.csv", index=False)
    curve = IB[IB.budget.notna()].groupby(["learner", "budget"]).norm_return.agg(["mean", "std"]); display(curve.unstack(0).round(3))
    tests = []
    for B in INTERACTION_BUDGETS:
        g = IB[IB.budget == B].groupby(["ds", "model", "learner"]).norm_return.mean().unstack()
        for other in ["REINFORCE", "PPO (1 epoch, no reuse)"]:
            tests.append(dict(budget=B, vs=other, mean_diff=float((g["PPO (10 epochs)"] - g[other]).mean()),
                              p=wilcoxon_p(g["PPO (10 epochs)"].values, g[other].values)))
    tests = pd.DataFrame(tests); tests["p_holm"] = holm(tests.p.values); display(tests.round(4)); tests.to_csv(f"{STUDY}/p4_tests.csv", index=False)
    small = tests[(tests.vs == "REINFORCE") & (tests.budget <= 5_000)]
    PPO_JUSTIFIED = bool(((small.mean_diff > 0) & (small.p_holm < 0.05)).any())
    print("Decision (pre-registered):", "PPO justified by sample efficiency under bandit feedback" if PPO_JUSTIFIED
          else "PPO not better than REINFORCE at small budgets; the paper states the learner is interchangeable")
    sdv = IB[IB.budget == max(INTERACTION_BUDGETS)].groupby(["learner", "ds", "model"]).norm_return.std().groupby("learner").mean()
    print("Mean seed SD of normalised return at the full budget:"); display(sdv.round(4))
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for nm in curve.index.get_level_values(0).unique():
        c_ = curve.loc[nm]; ax.plot(c_.index, c_["mean"], "o-", ms=3, label=nm)
        ax.fill_between(c_.index, c_["mean"] - c_["std"], c_["mean"] + c_["std"], alpha=0.15)
    ax.axhline(IB[IB.learner.str.startswith("Full")].norm_return.mean(), color="k", ls=":", label="Full information")
    ax.set_xscale("log"); ax.set_xlabel("Training episodes (bandit feedback)"); ax.set_ylabel("Normalised test return"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f"{STUDY}/fig_interaction_budget.pdf"); plt.show()

## P5. Ablations: features, start depth, prefix reuse, latency-cost reward

In [ ]:
if PROCEED and RUN["P5"]:
    AB = []
    for (ds, mk), C in CFG.items():
        specs = {"main": dict(groups=MAIN_G), "probe only": dict(groups=("probe",)), "probe+agree": dict(groups=("probe", "agree")),
                 "no probe (ret+query)": dict(groups=("ret", "query")), "start at depth 1": dict(groups=MAIN_G, start=1),
                 "no prefix reuse": dict(groups=MAIN_G, reuse=False), "latency-cost reward": dict(groups=MAIN_G, cost="lat", lam=LAT_LAMBDA_MAIN)}
        for nm, sp in specs.items():
            res = run_cascade(C, "abl", sp["groups"], sp.get("lam", LAMBDA_MAIN), FRONT_SEEDS, start=sp.get("start", 0),
                              cost=sp.get("cost", "tok"), reuse=sp.get("reuse", True), episodes=FRONT_EPISODES)
            AB.append(table_row(C, nm, res))
    AB = pd.DataFrame(AB); AB.to_csv(f"{STUDY}/p5_ablation.csv", index=False)
    display(AB.groupby("method")[["f1", "total_tokens", "latency_ms", "dF1_vs_mix_tok", "dF1_vs_mix_lat"]].mean().round(4))

## P6. Cross-dataset transfer and label-free recalibration (same backbone)

In [ ]:
if PROCEED and RUN["P6"]:
    TR = []
    for mk in PS_MODELS:
        for src in PS_DATASETS:
            for tgt in PS_DATASETS:
                if src == tgt or (src, mk) not in CFG or (tgt, mk) not in CFG: continue
                Cs, Ct = CFG[(src, mk)], CFG[(tgt, mk)]
                pca = hid_pca(Cs) if "hid" in MAIN_G else None
                def go():
                    Ds = cascade_data(Cs, MAIN_G, LAMBDA_MAIN, pca=pca); out = {}
                    norm_t = obs_norm(Ct, MAIN_G, 0, pca)                                     # label-free: target features only
                    for sd_ in FRONT_SEEDS:
                        actor, _ = train_cascade_ppo(Ds, sd_, episodes=FRONT_EPISODES)
                        for mode, nrm in [("source norm", Ds["norm"]), ("target norm", norm_t)]:
                            Ot = obs_tensor(Ct, "test", MAIN_G, 0, nrm, pca); st = greedy_cascade(actor, Ot, 0)
                            out[(mode, sd_)] = evaluate(Ct, "test", st, "cascade", n_dec=st + 1)
                    return out
                res = cached(f"{STUDY}/transfer/{mk}_{src}_to_{tgt}.pkl", go)
                indom = seed_avg([r["test"] for r in run_cascade(Ct, "front", MAIN_G, LAMBDA_MAIN, FRONT_SEEDS, episodes=FRONT_EPISODES).values()])
                for mode in ["source norm", "target norm"]:
                    avg = seed_avg([v for (m_, _), v in res.items() if m_ == mode])
                    d_mix, p_mix, _ = vs_frontier(Ct, avg, "total_tokens")
                    TR.append(dict(model=mk, src=src, tgt=tgt, mode=mode, f1=float(avg["f1"].mean()), f1_indomain=float(indom["f1"].mean()),
                                   dF1_vs_indomain=float((avg["f1"] - indom["f1"]).mean()), p_vs_indomain=wilcoxon_p(avg["f1"], indom["f1"]),
                                   tokens=float(avg["total_tokens"].mean()), tokens_indomain=float(indom["total_tokens"].mean()),
                                   dF1_vs_mix_tok=d_mix, p_vs_mix=p_mix))
    TR = pd.DataFrame(TR)
    if len(TR):
        TR["p_vs_indomain_holm"] = holm(TR.p_vs_indomain.values); TR.to_csv(f"{STUDY}/p6_transfer.csv", index=False)
        display(TR.round(4)); display(TR.groupby("mode")[["dF1_vs_indomain", "dF1_vs_mix_tok"]].mean().round(4))

## S. Claims sheet (every number used in the paper comes from here)

In [ ]:
if PROCEED and RUN["S"]:
    L = []
    def claim(s): L.append(s); print(s)
    claim(f"Configs: {len(CFG)} ({', '.join(f'{DNAME[d]}/{MNAME[m]}' for d, m in CFG)})")
    claim(f"Main feature set (validation-selected): {MAIN_FS} = {list(MAIN_G)}")
    for (ds, mk), C in CFG.items():
        claim(f"[{DNAME[ds]}/{MNAME[mk]}] decode-answer match with the generation cache: {C.match_rate}; "
              f"prompt-length match: {C.nin_match['test']:.3f}; latency model R2 prefill {C.LMOD['r2_prefill']:.3f}, decode {C.LMOD['r2_decode']:.3f}")
    if "P1" in globals():
        for nm, r in agg.iterrows():
            claim(f"[P1] {nm}: mean F1 {r.f1:.3f}, tokens {r.total_tokens:.0f}, latency {r.latency_ms:.0f} ms; dF1 vs static mixture "
                  f"at matched tokens {r.dF1_vs_mix_tok:+.4f} (sig. better {int(r.n_better_tok)}, sig. worse {int(r.n_worse_tok)} of {len(CFG)}); "
                  f"at matched latency {r.dF1_vs_mix_lat:+.4f} (better {int(r.n_better_lat)}, worse {int(r.n_worse_lat)})")
    if "area" in globals():
        for nm in area.index:
            claim(f"[P2] {nm}: mean dF1 above the static frontier along the lambda sweep: tokens {area.loc[nm, ('dF1_vs_mix_tok', 'mean')]:+.4f}, "
                  f"latency {area.loc[nm, ('dF1_vs_mix_lat', 'mean')]:+.4f}")
    if "BR" in globals() and len(BR):
        for (nm, fr), g in BR.groupby(["method", "frac"]):
            claim(f"[P3] {nm} budget {fr:.0%}: adherence {g.adherence.mean():.3f} (max {g.adherence.max():.3f}), dF1 vs val-calibrated mixture {g.dF1_vs_mix.mean():+.4f}")
    if "tests" in globals():
        claim(f"[P4] PPO justified at small budgets: {PPO_JUSTIFIED}")
        for _, r in tests.iterrows(): claim(f"[P4] budget {int(r.budget)}: PPO(10 ep) minus {r.vs} = {r.mean_diff:+.4f} (Holm p {r.p_holm:.4f})")
    if "AB" in globals():
        for nm, r in AB.groupby("method")[["f1", "total_tokens", "latency_ms", "dF1_vs_mix_tok", "dF1_vs_mix_lat"]].mean().iterrows():
            claim(f"[P5] {nm}: F1 {r.f1:.3f}, tokens {r.total_tokens:.0f}, latency {r.latency_ms:.0f}, dF1 vs mix tok {r.dF1_vs_mix_tok:+.4f}, lat {r.dF1_vs_mix_lat:+.4f}")
    if "TR" in globals() and len(TR):
        for mode, g in TR.groupby("mode"):
            claim(f"[P6] transfer ({mode}): mean dF1 vs in-domain {g.dF1_vs_indomain.mean():+.4f}, significant losses {int(((g.dF1_vs_indomain < 0) & (g.p_vs_indomain_holm < 0.05)).sum())}/{len(g)}")
    open(f"{STUDY}/claims.txt", "w").write("\n".join(L))
    if "P1" in globals():
        keep = ["Static k* (val F1)", "Adaptive-k (similarity gap)", "FLARE", "RL-AR (draft, paper)", "RL-AR-Lite (paper)",
                "Lite+rerank (reward regression, full info)", "Threshold cascade (agreement)", "Cascade (REINFORCE)",
                "Cascade (full information)", "RL-AR-Cascade (PPO, ours)", "Oracle depth (upper bound)"]
        t = P1[P1.method.isin(keep)].pivot_table(index=["ds", "model"], columns="method", values="dF1_vs_mix_tok")
        t.round(3).to_latex(f"{STUDY}/table_dF1_vs_mixture.tex"); P1.round(4).to_latex(f"{STUDY}/table_p1_full.tex", index=False)
    print("\nAll outputs in", STUDY)